In [1]:
!pip install music21

In [2]:
from google.colab import files
files.upload()  # Upload kaggle.json

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"sivaranjanimurugan","key":"aa3b204e69018be4fb8ae31f3f8aa199"}'}

In [3]:
!pip install kaggle

In [4]:
!kaggle datasets download -d imsparsh/musicnet-dataset

Dataset URL: https://www.kaggle.com/datasets/imsparsh/musicnet-dataset
License(s): CC0-1.0
100% 21.5G/21.5G [13:59<00:00, 27.5MB/s]



In [5]:
!kaggle datasets download -d imsparsh/musicnet-dataset

Dataset URL: https://www.kaggle.com/datasets/imsparsh/musicnet-dataset
License(s): CC0-1.0
musicnet-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
import os

for root, dirs, files in os.walk("dataset"):
    for file in files[:10]:
        print(os.path.join(root, file))

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
midi_path = "/content/drive/MyDrive/MIDI_Dataset/"

In [9]:
!pip install music21 tensorflow

In [10]:
import os

count = 0

for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))
        count += 1

        if count >= 30:
            break

    if count >= 30:
        break

In [11]:
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.mid') or file.endswith('.midi'):
            print(os.path.join(root, file))

In [12]:
!pip install music21

In [13]:
from music21 import stream, note

melody = stream.Stream()

notes_list = [
    'C4','E4','G4','C5',
    'G4','E4','C4',
    'D4','F4','A4','D5',
    'A4','F4','D4'
]

for n in notes_list:
    melody.append(note.Note(n))

melody.write('midi', fp='sample.mid')

print("sample.mid created")

sample.mid created


In [14]:
import os
print(os.listdir())

['.config', 'musicnet-dataset.zip', 'kaggle.json', 'sample.mid', 'drive', 'sample_data']


In [15]:
from music21 import converter, note

notes = []

midi = converter.parse("sample.mid")

for element in midi.flatten().notes:
    if isinstance(element, note.Note):
        notes.append(str(element.pitch))

print(notes)
print("Total Notes:", len(notes))

['C4', 'E4', 'G4', 'C5', 'G4', 'E4', 'C4', 'D4', 'F4', 'A4', 'D5', 'A4', 'F4', 'D4']
Total Notes: 14


In [19]:
import numpy as np
from tensorflow.keras.utils import to_categorical

pitchnames = sorted(set(notes))

note_to_int = {n:i for i,n in enumerate(pitchnames)}
int_to_note = {i:n for i,n in enumerate(pitchnames)}

sequence_length = 5

network_input = []
network_output = []

for i in range(len(notes)-sequence_length):
    network_input.append(
        [note_to_int[n] for n in notes[i:i+sequence_length]]
    )
    network_output.append(
        note_to_int[notes[i+sequence_length]]
    )

X = np.reshape(network_input,
               (len(network_input), sequence_length, 1))

X = X / float(len(pitchnames))

y = to_categorical(network_output, num_classes=len(pitchnames))

print(X.shape)
print(y.shape)

(9, 5, 1)
(9, 8)


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential()

model.add(
    LSTM(
        128,
        input_shape=(X.shape[1], X.shape[2])
    )
)

model.add(
    Dense(
        len(pitchnames),
        activation='softmax'
    )
)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam'
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 128)            │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,592 (264.03 KB)

 Trainable params: 67,592 (264.03 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
import numpy as np
from tensorflow.keras.utils import to_categorical

pitchnames = sorted(set(notes))

note_to_int = {n:i for i,n in enumerate(pitchnames)}
int_to_note = {i:n for i,n in enumerate(pitchnames)}

print("Unique Notes:", len(pitchnames))

sequence_length = 5

network_input = []
network_output = []

for i in range(len(notes)-sequence_length):
    seq_in = notes[i:i+sequence_length]
    seq_out = notes[i+sequence_length]

    network_input.append([note_to_int[n] for n in seq_in])
    network_output.append(note_to_int[seq_out])

X = np.reshape(
    network_input,
    (len(network_input), sequence_length, 1)
)

X = X / float(len(pitchnames))

y = to_categorical(
    network_output,
    num_classes=len(pitchnames)
)

print("X shape:", X.shape)
print("y shape:", y.shape)

Unique Notes: 8
X shape: (9, 5, 1)
y shape: (9, 8)


In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential()

model.add(
    LSTM(
        128,
        input_shape=(X.shape[1], X.shape[2])
    )
)

model.add(
    Dense(
        len(pitchnames),
        activation='softmax'
    )
)

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 128)            │        66,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 67,592 (264.03 KB)

 Trainable params: 67,592 (264.03 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.fit(
    X,
    y,
    epochs=50,
    batch_size=4
)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1111 - loss: 2.0911  
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.1111 - loss: 2.0673
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2222 - loss: 2.0556
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2222 - loss: 2.0423
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.2222 - loss: 2.0261
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.2222 - loss: 2.0142
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2222 - loss: 1.9985
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.2222 - loss: 1.9845
Epoch 9/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2222 - loss: 1.9667    
Epoch 10/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3333 - loss: 1.9501
Epoch 11/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3333 - loss: 1.9297
Epoch 12/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.2222 - loss: 1.

In [23]:
num_classes=len(pitchnames)

In [24]:
import random
import numpy as np

start = random.randint(0, len(network_input)-1)

pattern = network_input[start][:]

prediction_output = []

for i in range(20):

    prediction_input = np.reshape(
        pattern,
        (1, len(pattern), 1)
    )

    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(
        prediction_input,
        verbose=0
    )

    index = np.argmax(prediction)

    result = int_to_note[index]

    prediction_output.append(result)

    pattern.append(index)
    pattern = pattern[1:]

print(prediction_output)

['F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4', 'F4', 'A4']


In [25]:
from music21 import stream, note

offset = 0
output_notes = []

for pattern in prediction_output:

    new_note = note.Note(pattern)
    new_note.offset = offset

    output_notes.append(new_note)

    offset += 0.5

midi_stream = stream.Stream(output_notes)

midi_stream.write(
    'midi',
    fp='generated_music.mid'
)

print("generated_music.mid created")

generated_music.mid created


In [26]:
from google.colab import files

files.download("generated_music.mid")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
model.save("music_generator.h5")

In [28]:
from google.colab import files

model.save("music_generator.h5")

files.download("music_generator.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
from google.colab import files
files.download("music_generator.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>